# `compare.ipynb` — So sánh embedding (3 môn, align fine-tune)

Benchmark retrieval trên **3 giáo trình** (cùng domain với data train), metric **Recall@k** / **MRR@k**.

| Môn | PDF | Eval |
|-----|-----|------|
| `th` | Triết học Mác-Lênin | `training_data_th/test_data.json` |
| `pldc` | Pháp luật đại cương | `training_data_pldc/test_data.json` |
| `lsd` | Lịch sử Đảng | `training_data_lsd/test_data.json` |

Chỉ dùng **test split** (không eval trên train). Relevance: `positive` trong chunk (substring → token overlap ≥55%).

| Model | Vai trò |
|-------|---------|
| `bkai-base` | Trước fine-tune |
| `viet-bi-ft` | Sau MNRL |
| `bge-m3`, `e5-large` | Baseline ngoài |

**Chạy nhanh:** `RUN_ONLY = ["rag_chunk512"]`, `SUBJECTS_ONLY = ["lsd"]`.

Kết luận chính: **Δ fine-tune vs base** theo từng môn + **macro trung bình 3 môn**.


In [1]:
# CELL 1 — pip
%pip install -q langchain langchain-community langchain-chroma
%pip install -q langchain-text-splitters chromadb tiktoken python-dotenv PyMuPDF
%pip install -q -U sentence-transformers



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# CELL 2 — import
import gc
import json
import os
import re
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Set, Tuple

import numpy as np
import pandas as pd
import torch
import fitz

from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("OK import")


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OK import


In [3]:
# CELL 3 — config
IS_KAGGLE = Path("/kaggle").is_dir()

K_LIST = [1, 5, 10]
RETRIEVAL_K_MAX = max(K_LIST)
QA_EVAL_LIMIT: Optional[int] = None  # None = hết câu có relevant
TOKEN_OVERLAP_MIN = 0.55  # fallback khi substring không khớp chunk

VIET_BI_BASE_ID = "bkai-foundation-models/vietnamese-bi-encoder"

# Thí nghiệm: đổi RUN_ONLY để chạy nhanh (vd chỉ rag_chunk512)
EXPERIMENTS = [
    {
        "name": "rag_chunk512",
        "chunk_size": 512,
        "chunk_overlap": 64,
        "models": ["bkai-base", "viet-bi-ft", "bge-m3", "e5-large"],
    },
    {
        "name": "bkai_chunk256",
        "chunk_size": 256,
        "chunk_overlap": 32,
        "models": ["bkai-base", "viet-bi-ft"],
    },
]
RUN_ONLY: Optional[List[str]] = ["rag_chunk512"]  # vd ["rag_chunk512"] hoặc None = chạy hết

KAGGLE_DOCS_SLUG = "dangvy1507/unipolis-docs"
KAGGLE_MODEL_SLUG = "dangvy1507/viet-bi-encoder-ft"


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for d in [start, *start.parents]:
        td = d / "data" / "training_data" / "training_data_th" / "test_data.json"
        if td.is_file():
            return d
        pdf = d / "data" / "lich-su-dang-cong-san-viet-nam-giao-trinh.pdf"
        if pdf.is_file():
            return d
    return start


def build_subjects(repo_root: Path) -> List[Dict[str, Any]]:
    td = repo_root / "data" / "training_data"
    defs = [
        ("th", "Triết học", "triet-hoc-mac-lenin-giao-trinh.pdf", "training_data_th"),
        ("pldc", "Pháp luật đại cương", "phap-luat-dai-cuong-giao-trinh.pdf", "training_data_pldc"),
        ("lsd", "Lịch sử Đảng", "lich-su-dang-cong-san-viet-nam-giao-trinh.pdf", "training_data_lsd"),
    ]
    out = []
    for key, label, pdf_name, folder in defs:
        pdf = repo_root / "data" / pdf_name
        test_path = td / folder / "test_data.json"
        if not pdf.is_file():
            raise FileNotFoundError(f"Thiếu PDF: {pdf}")
        if not test_path.is_file():
            raise FileNotFoundError(f"Thiếu test_data: {test_path}")
        out.append({"key": key, "label": label, "pdf": pdf, "test_path": test_path})
    return out


KAGGLE_TRAINING_SLUG = "dangvy1507/training-data"
SUBJECTS_ONLY: Optional[List[str]] = None  # vd ["lsd"] hoặc None = cả 3 môn

if IS_KAGGLE:
    BASE = Path("/kaggle/input") / KAGGLE_DOCS_SLUG.split("/")[-1]
    MOD = Path("/kaggle/input") / KAGGLE_MODEL_SLUG.split("/")[-1]
    TRAIN_BASE = Path("/kaggle/input/datasets") / KAGGLE_TRAINING_SLUG.split("/")[-1]
    REPO_ROOT = Path("/kaggle/working")
    VIET_BI_FT_PATH = str(MOD / "vietnamese-bi-encoder-finetuned")
    CHROMA_BASE_DIR = "/kaggle/working/chroma_compare_nb"
    SUBJECTS = [
        {
            "key": "th",
            "label": "Triết học",
            "pdf": BASE / "triet-hoc-mac-lenin-giao-trinh.pdf",
            "test_path": TRAIN_BASE / "training_data_th" / "test_data.json",
        },
        {
            "key": "pldc",
            "label": "Pháp luật đại cương",
            "pdf": BASE / "phap-luat-dai-cuong-giao-trinh.pdf",
            "test_path": TRAIN_BASE / "training_data_pldc" / "test_data.json",
        },
        {
            "key": "lsd",
            "label": "Lịch sử Đảng",
            "pdf": BASE / "lich-su-dang-cong-san-viet-nam-giao-trinh.pdf",
            "test_path": TRAIN_BASE / "training_data_lsd" / "test_data.json",
        },
    ]
    for s in SUBJECTS:
        if not s["pdf"].is_file():
            raise FileNotFoundError(f"Kaggle thiếu PDF: {s['pdf']}")
        if not s["test_path"].is_file():
            raise FileNotFoundError(f"Kaggle thiếu test_data: {s['test_path']}")
    print("Kaggle")
else:
    from dotenv import load_dotenv

    REPO_ROOT = find_repo_root()
    for d in [Path.cwd().resolve(), REPO_ROOT, *REPO_ROOT.parents]:
        ef = d / ".env"
        if ef.is_file():
            load_dotenv(ef, override=False)
            break
    else:
        load_dotenv(override=False)

    _def_ft = REPO_ROOT / "models" / "vietnamese-bi-encoder-finetuned"
    VIET_BI_FT_PATH = (os.environ.get("VIET_BI_FT_PATH") or str(_def_ft)).strip()
    CHROMA_BASE_DIR = str(REPO_ROOT / "data" / "chroma_compare_nb")
    SUBJECTS = build_subjects(REPO_ROOT)
    print("Local | repo:", REPO_ROOT)

_p = Path(VIET_BI_FT_PATH)
_use = os.environ.get("VIET_BI_USE_PRETRAINED_IF_MISSING", "").lower() in ("1", "true", "yes")
_hf_id = not _p.is_absolute() and len(_p.parts) == 2 and not _p.is_dir()

if _p.is_dir():
    VIET_BI_FT_PATH = str(_p.resolve())
elif _hf_id:
    VIET_BI_FT_PATH = VIET_BI_FT_PATH.strip()
elif _use:
    VIET_BI_FT_PATH = VIET_BI_BASE_ID
    print("CANH BAO: khong tim thay fine-tuned local, dung base HF lam viet-bi-ft")
else:
    raise FileNotFoundError(
        f"Thieu model fine-tuned: {VIET_BI_FT_PATH}\n"
        "Dat VIET_BI_FT_PATH hoac train tu bi-encoder-2.ipynb"
    )

if RUN_ONLY:
    EXPERIMENTS = [e for e in EXPERIMENTS if e["name"] in RUN_ONLY]
if SUBJECTS_ONLY:
    SUBJECTS = [s for s in SUBJECTS if s["key"] in SUBJECTS_ONLY]

os.makedirs(CHROMA_BASE_DIR, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)
print("subjects:", [f"{s['key']} ({s['label']})" for s in SUBJECTS])
print("experiments:", [e["name"] for e in EXPERIMENTS])


Local | repo: /Users/dangvanvy/UniPolis
cpu
subjects: ['th (Triết học)', 'pldc (Pháp luật đại cương)', 'lsd (Lịch sử Đảng)']
experiments: ['rag_chunk512']


In [4]:
# CELL 4 — embeddings + cache
_emb: Dict[str, Embeddings] = {}

EMB = {
    "bkai-base": VIET_BI_BASE_ID,
    "viet-bi-ft": VIET_BI_FT_PATH,
    "bge-m3": "BAAI/bge-m3",
    "e5-large": "intfloat/multilingual-e5-large",
}


class E5Wrap(Embeddings):
    def __init__(self, inner: HuggingFaceEmbeddings):
        self._i = inner

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        p = [t if t.lstrip().lower().startswith("passage:") else f"passage: {t}" for t in texts]
        return self._i.embed_documents(p)

    def embed_query(self, text: str) -> List[float]:
        t = text if text.lstrip().lower().startswith("query:") else f"query: {text}"
        return self._i.embed_documents([t])[0]


def get_emb(model_id: str) -> Embeddings:
    if model_id not in _emb:
        print("load", model_id[:56], "...")
        base = HuggingFaceEmbeddings(
            model_name=model_id,
            model_kwargs={"device": DEVICE},
            encode_kwargs={"normalize_embeddings": True},
        )
        if "multilingual-e5" in model_id.lower() or "/e5-" in model_id.lower():
            _emb[model_id] = E5Wrap(base)
        else:
            _emb[model_id] = base
    return _emb[model_id]


def unload_emb(model_id: str) -> None:
    if model_id in _emb:
        del _emb[model_id]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print({k: EMB[k] for k in EMB})


{'bkai-base': 'bkai-foundation-models/vietnamese-bi-encoder', 'viet-bi-ft': '/Users/dangvanvy/UniPolis/models/vietnamese-bi-encoder-finetuned', 'bge-m3': 'BAAI/bge-m3', 'e5-large': 'intfloat/multilingual-e5-large'}


In [5]:
# CELL 5 — PDF + test_data (3 môn)


def clean_text(t: str) -> str:
    t = re.sub(r"\s+", " ", t)
    return t.strip()


def load_pdf(path: str) -> List[Document]:
    out = []
    with fitz.open(path) as pdf:
        for i in range(len(pdf)):
            t = clean_text(pdf[i].get_text())
            if t:
                out.append(Document(page_content=t, metadata={"page": i + 1}))
    return out


def load_test_items(path: Path) -> List[Dict]:
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError(f"test_data phải là list: {path}")
    items = []
    for i, row in enumerate(data):
        items.append({
            "id": row.get("positive_id") or f"q{i}",
            "question": row["query"],
            "positive": row.get("positive", ""),
        })
    return items


subject_docs: Dict[str, List[Document]] = {}
subject_tests: Dict[str, List[Dict]] = {}

for sub in SUBJECTS:
    sk = sub["key"]
    subject_docs[sk] = load_pdf(str(sub["pdf"]))
    subject_tests[sk] = load_test_items(sub["test_path"])
    print(
        f"{sk:5} {sub['label']:22} | pages {len(subject_docs[sk]):3} "
        f"| test {len(subject_tests[sk]):3}"
    )


th    Triết học              | pages 214 | test 144
pldc  Pháp luật đại cương    | pages 238 | test 140
lsd   Lịch sử Đảng           | pages 223 | test 143


In [6]:
# CELL 6 — chunk helpers


def build_chunks(docs: List[Document], chunk_size: int, chunk_overlap: int) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " ", ""],
    )
    out = splitter.split_documents(docs)
    for i, d in enumerate(out):
        d.metadata = dict(d.metadata or {})
        d.metadata["chunk_id"] = str(i)
    return out


In [7]:
# CELL 7 — relevance (substring + token overlap fallback)


def norm(s: str) -> str:
    return re.sub(r"\s+", " ", s.lower().strip())


def snippet(it: Dict) -> str:
    p = (it.get("positive") or "").strip()
    if len(norm(p)) >= 12:
        return p
    e = (it.get("expected_chunk_content") or "").strip()
    if len(norm(e)) >= 12:
        return e
    g = (it.get("ground_truth_answer") or "").strip()
    return g[:120] if g else ""


def token_set(s: str) -> Set[str]:
    return set(re.findall(r"\w+", norm(s), flags=re.UNICODE))


def rel_ids(
    it: Dict,
    chs: List[Document],
    min_overlap: float = TOKEN_OVERLAP_MIN,
) -> Tuple[Set[str], str]:
    nd = norm(snippet(it))
    if len(nd) < 8:
        return set(), "no_snippet"

    found: Set[str] = set()
    for d in chs:
        cid = d.metadata.get("chunk_id")
        if cid is not None and nd in norm(d.page_content):
            found.add(str(cid))
    if found:
        return found, "substring"

    qtok = token_set(nd)
    if len(qtok) < 4:
        return set(), "snippet_too_short"

    for d in chs:
        cid = d.metadata.get("chunk_id")
        if cid is None:
            continue
        ctok = token_set(d.page_content)
        if not ctok:
            continue
        if len(qtok & ctok) / len(qtok) >= min_overlap:
            found.add(str(cid))
    if found:
        return found, "token_overlap"
    return set(), "no_match"


def build_eval_qa(
    items: List[Dict],
    chs: List[Document],
) -> Tuple[List[Dict], Dict[str, Set[str]], Dict[str, int]]:
    qa: List[Dict] = []
    rel_map: Dict[str, Set[str]] = {}
    stats = {"substring": 0, "token_overlap": 0, "skip": 0}

    for it in items:
        rid = str(it.get("id", f"q{len(qa)}"))
        r, how = rel_ids(it, chs)
        if not r:
            stats["skip"] += 1
            continue
        it = dict(it)
        it["_rid"] = rid
        it["_match"] = how
        qa.append(it)
        rel_map[rid] = r
        stats[how] = stats.get(how, 0) + 1

    if QA_EVAL_LIMIT and QA_EVAL_LIMIT > 0:
        qa = qa[: int(QA_EVAL_LIMIT)]
        rel_map = {it["_rid"]: rel_map[it["_rid"]] for it in qa}

    return qa, rel_map, stats


In [8]:
# CELL 8 — Recall@k, MRR@k + chạy theo từng experiment


def recall_k(ranked: List[str], rel: Set[str], k: int) -> float:
    if not rel:
        return 0.0
    return len(rel & set(ranked[:k])) / len(rel)


def mrr_k(ranked: List[str], rel: Set[str], k: int) -> float:
    for i, x in enumerate(ranked[:k], start=1):
        if x in rel:
            return 1.0 / i
    return 0.0


def ranked(vs: Chroma, q: str, k: int) -> Tuple[List[str], float]:
    t0 = time.time()
    hits = vs.similarity_search(q, k=k)
    dt = time.time() - t0
    ids = []
    for d in hits:
        cid = (d.metadata or {}).get("chunk_id")
        if cid is not None:
            ids.append(str(cid))
    return ids, dt


def run_key(
    key: str,
    exp_name: str,
    sub_key: str,
    sub_label: str,
    chunks: List[Document],
    qa: List[Dict],
    rel_map: Dict[str, Set[str]],
) -> Dict[str, Any]:
    mid = EMB[key]
    col = f"{exp_name}_{sub_key}_{key}"
    vs = Chroma.from_documents(
        documents=chunks,
        embedding=get_emb(mid),
        persist_directory=os.path.join(CHROMA_BASE_DIR, col),
        ids=[d.metadata["chunk_id"] for d in chunks],
    )
    rows, lats = [], []
    for it in qa:
        rid = it["_rid"]
        rel = rel_map[rid]
        q = it.get("question") or it.get("query", "")
        rnk, dt = ranked(vs, q, RETRIEVAL_K_MAX)
        lats.append(dt)
        row = {"rid": rid, "lat": dt}
        for kk in K_LIST:
            row[f"R@{kk}"] = recall_k(rnk, rel, kk)
            row[f"M@{kk}"] = mrr_k(rnk, rel, kk)
        rows.append(row)
    vs.delete_collection()
    del vs
    unload_emb(mid)

    df = pd.DataFrame(rows)
    out = {
        "experiment": exp_name,
        "subject": sub_key,
        "subject_label": sub_label,
        "key": key,
        "model": mid,
        "n": len(qa),
        "mean_lat": float(np.mean(lats)),
    }
    for kk in K_LIST:
        out[f"recall@{kk}"] = float(df[f"R@{kk}"].mean())
        out[f"mrr@{kk}"] = float(df[f"M@{kk}"].mean())
    return out


all_results: List[Dict[str, Any]] = []
eval_reports: List[Dict[str, Any]] = []

for exp in EXPERIMENTS:
    name = exp["name"]
    cs, co = exp["chunk_size"], exp["chunk_overlap"]
    print(f"\n{'=' * 72}\nEXPERIMENT: {name} (chunk {cs}/{co})\n{'=' * 72}")

    for sub in SUBJECTS:
        sk, slabel = sub["key"], sub["label"]
        raw = subject_docs[sk]
        qa_items = subject_tests[sk]
        chunks = build_chunks(raw, cs, co)
        qa, rel_map, rel_stats = build_eval_qa(qa_items, chunks)

        report = {
            "experiment": name,
            "subject": sk,
            "subject_label": slabel,
            "chunk_size": cs,
            "chunk_overlap": co,
            "chunks": len(chunks),
            "test_total": len(qa_items),
            "eval": len(qa),
            **rel_stats,
        }
        eval_reports.append(report)
        print(
            f"\n  [{sk}] {slabel} | chunks={len(chunks)} | "
            f"eval={len(qa)}/{len(qa_items)} skip={rel_stats['skip']} | "
            f"sub={rel_stats.get('substring', 0)} tok={rel_stats.get('token_overlap', 0)}"
        )
        if not qa:
            print(f"    BO QUA: khong co cau co relevant chunk")
            continue

        for key in exp["models"]:
            if key not in EMB:
                raise KeyError(f"Model key khong co trong EMB: {key}")
            print(f"    run {key} ...")
            all_results.append(
                run_key(key, name, sk, slabel, chunks, qa, rel_map)
            )

print("\ndone |", len(all_results), "runs")



EXPERIMENT: rag_chunk512 (chunk 512/64)

  [th] Triết học | chunks=1448 | eval=136/144 skip=8 | sub=10 tok=126
    run bkai-base ...
load bkai-foundation-models/vietnamese-bi-encoder ...


/var/folders/06/55ncynjj69l30_gnnsnw6xv40000gn/T/ipykernel_64508/324792318.py:28: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  base = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9222.73it/s]


    run viet-bi-ft ...
load /Users/dangvanvy/UniPolis/models/vietnamese-bi-encoder-f ...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7588.50it/s]


    run bge-m3 ...
load BAAI/bge-m3 ...


...

    run bge-m3 ...
load BAAI/bge-m3 ...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 65036.99it/s]


    run e5-large ...
load intfloat/multilingual-e5-large ...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 5884.81it/s]



done | 12 runs


In [9]:
# CELL 9 — bảng theo môn + macro avg + delta fine-tune


def weighted_mean(part: pd.DataFrame, col: str) -> float:
    if part.empty or part["n"].sum() == 0:
        return 0.0
    return float((part[col] * part["n"]).sum() / part["n"].sum())


rows = []
for r in all_results:
    row = {
        "experiment": r["experiment"],
        "subject": r["subject"],
        "subject_label": r["subject_label"],
        "embedding": r["key"],
        "model": r["model"],
        "n": r["n"],
        "lat_s": r["mean_lat"],
    }
    for kk in K_LIST:
        row[f"Recall@{kk}"] = r[f"recall@{kk}"]
        row[f"MRR@{kk}"] = r[f"mrr@{kk}"]
    rows.append(row)

df = pd.DataFrame(rows)
km = max(K_LIST)
macro_rows: List[Dict[str, Any]] = []

for exp_name in df["experiment"].unique():
    edf = df[df["experiment"] == exp_name]
    print("\n" + "=" * 72)
    print("EXPERIMENT:", exp_name)
    print("=" * 72)

    for sk in sorted(edf["subject"].unique()):
        sub = edf[edf["subject"] == sk].copy()
        sub = sub.sort_values([f"Recall@{km}", f"MRR@{km}"], ascending=False)
        label = sub["subject_label"].iloc[0]
        print(f"\n--- {sk.upper()} | {label} ---")
        cols = ["embedding", "n", "lat_s", f"Recall@{km}", f"MRR@{km}"]
        print(sub[cols].to_string(index=False))

        base = sub[sub["embedding"] == "bkai-base"]
        ft = sub[sub["embedding"] == "viet-bi-ft"]
        if len(base) and len(ft):
            d_mrr = float(ft[f"MRR@{km}"].iloc[0] - base[f"MRR@{km}"].iloc[0])
            d_rec = float(ft[f"Recall@{km}"].iloc[0] - base[f"Recall@{km}"].iloc[0])
            print(f"  Fine-tune vs base: MRR@{km} {d_mrr:+.4f} | Recall@{km} {d_rec:+.4f}")

    print(f"\n--- MACRO (trung binh co trong so n, {len(edf['subject'].unique())} mon) ---")
    macro_part = []
    for emb in ["bkai-base", "viet-bi-ft", "bge-m3", "e5-large"]:
        part = edf[edf["embedding"] == emb]
        if part.empty:
            continue
        mrow = {
            "experiment": exp_name,
            "subject": "macro",
            "embedding": emb,
            "n": int(part["n"].sum()),
            f"Recall@{km}": weighted_mean(part, f"Recall@{km}"),
            f"MRR@{km}": weighted_mean(part, f"MRR@{km}"),
        }
        macro_rows.append(mrow)
        macro_part.append(mrow)
    mdf = pd.DataFrame(macro_part)
    if not mdf.empty:
        print(mdf[["embedding", "n", f"Recall@{km}", f"MRR@{km}"]].to_string(index=False))
        b = mdf[mdf["embedding"] == "bkai-base"]
        f = mdf[mdf["embedding"] == "viet-bi-ft"]
        if len(b) and len(f):
            d_mrr = float(f[f"MRR@{km}"].iloc[0] - b[f"MRR@{km}"].iloc[0])
            d_rec = float(f[f"Recall@{km}"].iloc[0] - b[f"Recall@{km}"].iloc[0])
            print(f"\n>>> KET LUAN FINE-TUNE (macro 3 mon): MRR@{km} {d_mrr:+.4f} | Recall@{km} {d_rec:+.4f}")

print("\n--- Eval / relevance per subject ---")
print(pd.DataFrame(eval_reports).to_string(index=False))



EXPERIMENT: rag_chunk512

--- LSD | Lịch sử Đảng ---
 embedding   n    lat_s  Recall@10   MRR@10
    bge-m3 125 0.118485   0.721333 0.510895
  e5-large 125 0.120464   0.709333 0.524305
viet-bi-ft 125 0.040080   0.689333 0.511089
 bkai-base 125 0.038828   0.534667 0.372987
  Fine-tune vs base: MRR@10 +0.1381 | Recall@10 +0.1547

--- PLDC | Pháp luật đại cương ---
 embedding   n    lat_s  Recall@10   MRR@10
    bge-m3 131 0.102035   0.792939 0.667712
viet-bi-ft 131 0.038775   0.777036 0.629259
  e5-large 131 0.117721   0.764631 0.693236
 bkai-base 131 0.061725   0.669847 0.557927
  Fine-tune vs base: MRR@10 +0.0713 | Recall@10 +0.1072

--- TH | Triết học ---
 embedding   n    lat_s  Recall@10   MRR@10
  e5-large 136 0.129231   0.761029 0.584512
viet-bi-ft 136 0.038892   0.757353 0.599650
    bge-m3 136 0.119911   0.746324 0.594552
 bkai-base 136 0.040466   0.606618 0.441427
  Fine-tune vs base: MRR@10 +0.1582 | Recall@10 +0.1507

--- MACRO (trung binh co trong so n, 3 mon) ---
 embeddin

In [10]:
# CELL 10 — lưu
import datetime
import platform

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
outd = Path("/kaggle/working") if IS_KAGGLE else REPO_ROOT
outd.mkdir(parents=True, exist_ok=True)
csv_p = outd / f"compare_retrieval_{ts}.csv"
df.to_csv(csv_p, index=False, encoding="utf-8-sig")

ft_deltas = []
km = max(K_LIST)
for exp_name in df["experiment"].unique():
    for sk in df["subject"].unique():
        sub = df[(df["experiment"] == exp_name) & (df["subject"] == sk)]
        base = sub[sub["embedding"] == "bkai-base"]
        ft = sub[sub["embedding"] == "viet-bi-ft"]
        if len(base) and len(ft):
            ft_deltas.append({
                "experiment": exp_name,
                "subject": sk,
                "delta_mrr": float(ft[f"MRR@{km}"].iloc[0] - base[f"MRR@{km}"].iloc[0]),
                "delta_recall": float(ft[f"Recall@{km}"].iloc[0] - base[f"Recall@{km}"].iloc[0]),
            })
    # macro delta
    edf = df[df["experiment"] == exp_name]
    def _wmean(emb):
        p = edf[edf["embedding"] == emb]
        return weighted_mean(p, f"MRR@{km}") if len(p) else None
    b_m, f_m = _wmean("bkai-base"), _wmean("viet-bi-ft")
    if b_m is not None and f_m is not None:
        ft_deltas.append({
            "experiment": exp_name,
            "subject": "macro",
            "delta_mrr": float(f_m - b_m),
            "delta_recall": float(
                weighted_mean(edf[edf["embedding"] == "viet-bi-ft"], f"Recall@{km}")
                - weighted_mean(edf[edf["embedding"] == "bkai-base"], f"Recall@{km}")
            ),
        })

meta = {
    "ts": ts,
    "k_list": K_LIST,
    "retrieval_k_max": RETRIEVAL_K_MAX,
    "token_overlap_min": TOKEN_OVERLAP_MIN,
    "subjects": [{"key": s["key"], "label": s["label"], "pdf": str(s["pdf"]), "test": str(s["test_path"])} for s in SUBJECTS],
    "experiments": EXPERIMENTS,
    "eval_reports": eval_reports,
    "finetune_vs_base": ft_deltas,
    "macro_summary": macro_rows,
    "device": DEVICE,
    "python": platform.python_version(),
    "rows": rows,
}
js_p = outd / f"compare_retrieval_{ts}.json"
with open(js_p, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print(csv_p)
print(js_p)


/Users/dangvanvy/UniPolis/compare_retrieval_20260520_013058.csv
/Users/dangvanvy/UniPolis/compare_retrieval_20260520_013058.json


## Ghi chú

- **Eval:** `test_data.json` từng môn (test split), relevance từ field `positive`.
- **Kết luận chính:** dòng `>>> KET LUAN FINE-TUNE (macro 3 mon)` — không dùng MRR val của `bi-encoder-2.ipynb`.
- Chạy nhanh: `RUN_ONLY = ["rag_chunk512"]`, `SUBJECTS_ONLY = ["lsd"]`.
- Chunk 512/64 (RAG) vs 256/32 (ablation BKAI). Metric chỉ retrieval.
